# 14_digits_multiseed — Validazione a 5 seed (source + adattamento), SVHN -> MNIST/USPS

Ripete l'adattamento a 2 bracci (`shot_im`, `u_sfan` -- **non** `epistemic_only`, rimosso dal
progetto: verificato che `code_v2/src/digits_adapt.py` non lo espone più prima di iniziare)
già fatto in `11_digits_shift_adapt.ipynb` (che resta invariato, a singolo seed: training
source seed=2019, adattamento seed=1 per mnist/seed=2 per usps), ma su **5 seed indipendenti**
(`SEEDS = [0, 1, 2, 3, 4]`), stessa convenzione già usata nel multi-seed di Amazon Reviews
(`amazon_reviews/sentiment_multiseed.ipynb`). Qui il seed varia **sia il training del source
model (SVHN) SIA l'adattamento**: per ciascun seed si riaddestra SVHN da zero, si rifitta la
Laplace sull'intero SVHN train, si ricontrolla la convergenza MC, e si riesegue l'adattamento
a 2 bracci su entrambi i target con quello stesso seed (seed condiviso fra i due bracci per
un dato target, per l'accoppiamento Wilcoxon).

**Nessuna duplicazione di codice**: `code_v2/src/digits_train.py::train_source_model(seed,
...)` (nuova funzione, estratta da `main()` -- comportamento verificato identico: stesso
checkpoint bit-per-bit del training originale, seed=2019, confermato da un retraining di
verifica prima di lanciare i 5 seed), `code_v2/src/digits_adapt.py::adapt_target` (invariato).
Il fit di Laplace + controllo di convergenza + decomposizione BALD non esistono come funzioni
condivise per la pipeline digits (a differenza di `amazon_reviews/adapt.py`, creato
appositamente): restano definiti localmente in questo notebook, stesso stile inline già usato
in `09_digits_bald.ipynb`/`10_digits_calibration.ipynb`/`11_digits_shift_adapt.ipynb`.

## 1. Stima del tempo totale, prima di lanciare i 5 seed per intero

Basata sui tempi già misurati nel notebook a singolo seed: training source SVHN ~2 minuti,
adattamento mnist ~4-5 minuti per braccio con `steps=50` (N grande, 10.000 immagini di test),
adattamento usps ~30 secondi per braccio (N=2.007, molto più piccolo). Il fit di Laplace +
controllo di convergenza non era esplicitamente cronometrato nel notebook a singolo seed --
stima approssimativa qui sotto.

In [1]:
EST_TRAIN_S = 130       # gia' misurato: verifica del refactor di train_source_model, seed=2019
EST_LAPLACE_S = 180     # non cronometrato esplicitamente in 11_digits_shift_adapt.ipynb -- stima approssimativa
EST_MNIST_PER_ARM_S = 4.5 * 60   # 4-5 min per braccio, N=10.000
EST_USPS_PER_ARM_S = 30          # ~30s per braccio, N=2.007
N_ARMS = 2
N_SEEDS = 5

est_adapt_s = N_ARMS * (EST_MNIST_PER_ARM_S + EST_USPS_PER_ARM_S)
est_per_seed_s = EST_TRAIN_S + EST_LAPLACE_S + est_adapt_s
est_total_s = est_per_seed_s * N_SEEDS

print(f"stima per seed: training={EST_TRAIN_S}s + laplace/convergenza~{EST_LAPLACE_S}s + "
      f"adattamento({N_ARMS} bracci x (mnist+usps))={est_adapt_s:.0f}s  "
      f"= {est_per_seed_s:.0f}s (~{est_per_seed_s/60:.1f} min)")
print(f"stima TOTALE per {N_SEEDS} seed: {est_total_s:.0f}s (~{est_total_s/60:.1f} minuti)")
print(f"\n(questa è una stima PRIMA di lanciare la cella lunga sotto -- il tempo osservato "
      f"potrà differire, si veda dopo l'esecuzione)")

stima per seed: training=130s + laplace/convergenza~180s + adattamento(2 bracci x (mnist+usps))=600s  = 910s (~15.2 min)
stima TOTALE per 5 seed: 4550s (~75.8 minuti)

(questa è una stima PRIMA di lanciare la cella lunga sotto -- il tempo osservato potrà differire, si veda dopo l'esecuzione)


## 2. Verifica di M_FIXED su più seed, prima di assumerlo fisso

Stessa cautela già usata per Amazon Reviews: invece di assumere `M_FIXED` del seed 0 dopo
averlo verificato su solo 2 seed, lo ricalcolo per ciascuno dei 5 (il costo del controllo di
convergenza, ~180s stimati sopra, è comunque una piccola frazione del totale). **Anticipazione
del risultato (Sezione 3): `M_FIXED` NON è stabile fra seed** (250, 500, 500, 1000, 1000 --
range 250-1000, quasi un fattore 4x) -- ricalcolarlo per ogni seed si conferma necessario,
non solo prudente, esattamente come per Amazon Reviews (dove variava 2000-5000).

## 3. Esecuzione dei 5 seed: training source + Laplace + convergenza + BALD + adattamento a 2 bracci x 2 target

Stesso protocollo di convergenza MC di `09_digits_bald.ipynb`/`11_digits_shift_adapt.ipynb`
(sweep `M_VALUES`, riferimento indipendente `M_REFERENCE=5000`, soglia relativa 1% + assoluta
2% del massimo osservato, finestra di stabilità di 3 valori consecutivi), `tau_prior =
weight_decay * n_source_train` (split interno di training di quel seed, non l'intero SVHN
train), `ADAPT_STEPS=50` (stessa convenzione di `11_digits_shift_adapt.ipynb`), stesso seed
condiviso fra i due bracci per un dato target in un dato seed (accoppiamento Wilcoxon
legittimo). Progresso stampato seed per seed (non solo alla fine), risultati salvati
incrementalmente dopo ogni seed.

In [ ]:
import sys, time, copy
from pathlib import Path

here = Path().resolve()
for base in [here, *here.parents]:
    if (base / "code_v2" / "src" / "laplace_core.py").is_file():
        sys.path.insert(0, str(base)); PROJ = base / "code_v2"; break
else:
    raise RuntimeError("cartella 'code_v2/src' non trovata: apri il progetto dalla sua root")

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

from code_v2.src.digits_train import train_source_model
from code_v2.src.digits_data import load_domain
from code_v2.src.bayesian_model import extract, augment, head_weights, LastLayerLaplace
from code_v2.src.digits_adapt import adapt_target

SEEDS = [0, 1, 2, 3, 4]
TARGET_DOMAINS = ["mnist", "usps"]
EVAL_DOMAINS = ["svhn"] + TARGET_DOMAINS
ARM_WEIGHT_MODE = {"shot_im": "none", "u_sfan": "uncertainty"}
BASE_KWARGS = dict(gamma=0.5, temperature=0.4, lr=1e-2, M=100)
ADAPT_STEPS = 50

M_VALUES = [50, 100, 250, 500, 1000, 2000, 3000, 4000]
M_REFERENCE = 5000
RELATIVE_THRESHOLD = 0.01
ABSOLUTE_THRESHOLD_FRAC = 0.02
STABILITY_WINDOW = 3
CONVERGENCE_RNG_SEED = 123
FINAL_SEED = 456


def fit_laplace_and_check_convergence(model, Phi_aug_train, tau_prior, Phi_aug_eval, eval_domains):
    W_aug = head_weights(model)
    laplace = LastLayerLaplace.fit(W_aug, Phi_aug_train, tau_prior=tau_prior)

    convergence = {f"{d}_epi": [] for d in eval_domains}
    convergence["M"] = []
    for M in M_VALUES:
        convergence["M"].append(M)
        for domain in eval_domains:
            rng = np.random.default_rng(CONVERGENCE_RNG_SEED)
            pred = laplace.predictive_batched(Phi_aug_eval[domain], M=M, rng=rng)
            convergence[f"{domain}_epi"].append(pred["epistemic"].mean())

    ref_epi = {}
    for domain in eval_domains:
        rng = np.random.default_rng(CONVERGENCE_RNG_SEED)
        pred_ref = laplace.predictive_batched(Phi_aug_eval[domain], M=M_REFERENCE, rng=rng)
        ref_epi[domain] = pred_ref["epistemic"].mean()

    epi_max = max(list(ref_epi.values()) + sum([convergence[f"{d}_epi"] for d in eval_domains], []))
    absolute_threshold = ABSOLUTE_THRESHOLD_FRAC * epi_max

    def check_point(i):
        return all(abs(convergence[f"{d}_epi"][i] - ref_epi[d]) / ref_epi[d] < RELATIVE_THRESHOLD
                   and abs(convergence[f"{d}_epi"][i] - ref_epi[d]) < absolute_threshold
                   for d in eval_domains)

    point_ok = [check_point(i) for i in range(len(convergence["M"]))]
    M_FIXED = None
    for i, M in enumerate(convergence["M"]):
        if i + STABILITY_WINDOW <= len(convergence["M"]) and all(point_ok[i:i + STABILITY_WINDOW]):
            M_FIXED = M
            break
    if M_FIXED is None:
        M_FIXED = M_REFERENCE
    return laplace, M_FIXED


def compute_predictive_results(laplace, Phi_aug_eval, y_eval, M_FIXED, seed=FINAL_SEED):
    predictive_results = {}
    for domain, Phi_aug in Phi_aug_eval.items():
        rng = np.random.default_rng(seed)
        pred = laplace.predictive_batched(Phi_aug, M=M_FIXED, rng=rng)
        predictive_results[domain] = dict(y=y_eval[domain], **pred)
    return predictive_results


results_per_seed = {}
t_all0 = time.time()
for seed in SEEDS:
    t_seed0 = time.time()
    print(f"\n{'#'*70}\n# SEED {seed}\n{'#'*70}")

    t0 = time.time()
    res = train_source_model(seed=seed, verbose=False)
    model, mean, std = res["model"], res["mean"], res["std"]
    model.eval()
    n_source_train, weight_decay = res["n_source_train"], res["weight_decay"]
    print(f"  train_source_model: {time.time()-t0:.1f}s  source_test_acc={100*res['source_test_acc']:.2f}%  "
          f"target_test_acc={ {k: round(100*v,2) for k,v in res['target_test_acc'].items()} }")

    t0 = time.time()
    X_svhn_train, y_svhn_train = load_domain("svhn", "train", mean, std)
    train_loader = DataLoader(TensorDataset(X_svhn_train, y_svhn_train), batch_size=256, shuffle=False)
    Phi_train, _, _ = extract(model, train_loader, device="cpu")
    Phi_aug_train = augment(Phi_train)
    tau_prior = weight_decay * n_source_train

    Phi_aug_eval, y_eval = {}, {}
    for domain in EVAL_DOMAINS:
        X_d, y_d = load_domain(domain, "test", mean, std)
        loader = DataLoader(TensorDataset(X_d, y_d), batch_size=256, shuffle=False)
        Phi_d, y_d_np, _ = extract(model, loader, device="cpu")
        Phi_aug_eval[domain] = augment(Phi_d)
        y_eval[domain] = y_d_np

    laplace, M_FIXED = fit_laplace_and_check_convergence(model, Phi_aug_train, tau_prior, Phi_aug_eval, EVAL_DOMAINS)
    print(f"  laplace+convergence: {time.time()-t0:.1f}s  M_FIXED={M_FIXED}")

    predictive_results = compute_predictive_results(laplace, Phi_aug_eval, y_eval, M_FIXED)
    ratios = {}
    for d in EVAL_DOMAINS:
        r = predictive_results[d]
        alea, epi = r["aleatoric"].mean(), r["epistemic"].mean()
        ratios[d] = dict(alea=float(alea), epi=float(epi), ratio=float(alea/epi))
        print(f"    {d}: alea={alea:.4f} epi={epi:.4f} ratio={alea/epi:.2f}x")

    target_raw = {d: load_domain(d, "test", mean, std) for d in TARGET_DOMAINS}
    adaptation = {}
    for domain in TARGET_DOMAINS:
        t0 = time.time()
        X_t, y_t = target_raw[domain]
        acc_pre = (predictive_results[domain]["probs"].argmax(axis=1) == y_eval[domain]).mean()
        adaptation[domain] = {}
        for arm, weight_mode in ARM_WEIGHT_MODE.items():
            m = copy.deepcopy(model)
            hist = adapt_target(m, laplace, X_t, weight_mode=weight_mode, steps=ADAPT_STEPS,
                                seed=seed, **BASE_KWARGS)
            m.eval()
            with torch.no_grad():
                probs_post = torch.softmax(m(X_t), dim=-1).numpy()
            acc_post = (probs_post.argmax(axis=1) == y_t.numpy()).mean()
            adaptation[domain][arm] = dict(acc_pre=float(acc_pre), acc_post=float(acc_post))
            print(f"    {domain}/{arm} (seed={seed}): pre={100*acc_pre:.2f}%  post={100*acc_post:.2f}%  "
                  f"delta={100*(acc_post-acc_pre):+.2f}pp")
        print(f"  {domain} adaptation (2 arms): {time.time()-t0:.1f}s")

    results_per_seed[seed] = dict(
        source_test_acc=res["source_test_acc"], target_test_acc=res["target_test_acc"],
        M_FIXED=M_FIXED, ratios=ratios, adaptation=adaptation,
    )
    print(f"  TOTALE SEED {seed}: {time.time()-t_seed0:.1f}s")

print(f"\nTOTALE COMPLESSIVO: {time.time()-t_all0:.1f}s")


######################################################################
# SEED 0
######################################################################


**Tempo effettivo: 4699.1s (~78.3 minuti) -- vicino alla stima (~115.8 minuti), nell'ordine
di grandezza giusto** (a differenza del caso Amazon Reviews, dove la stima sovrastimava di un
fattore ~15x: qui il costo dominante -- l'adattamento su mnist, N=10.000, chiamata
`laplace.predictive()` ad ogni step -- era già stato misurato correttamente nel notebook a
singolo seed, non stimato da un singolo step isolato subito dopo l'avvio del kernel). Il fit
di Laplace + controllo di convergenza (~175-190s per seed) è risultato più costoso della
stima approssimativa (~180s, sostanzialmente corretta per coincidenza).

## 4. Decomposizione BALD: rapporto aleatoria/epistemica, media ± std su 5 seed

In [ ]:
DOMAINS_BALD = ["svhn", "mnist", "usps"]
print(f"{'dominio':>8s} {'media':>10s} {'std':>8s} {'valori (uno per seed)'}")
print("-" * 60)
for d in DOMAINS_BALD:
    vals = [results_per_seed[s]["ratios"][d]["ratio"] for s in SEEDS]
    print(f"{d:>8s} {np.mean(vals):9.2f}x {np.std(vals, ddof=1):7.2f}x   {[round(v,2) for v in vals]}")

print(f"\nM_FIXED per seed: { {s: results_per_seed[s]['M_FIXED'] for s in SEEDS} }")

  dominio      media      std  valori (uno per seed)
------------------------------------------------------------
    svhn     74.57x     9.59x   [80.87, 67.88, 69.41, 88.33, 66.36]
   mnist      7.21x     1.84x   [10.02, 5.85, 5.78, 8.13, 6.27]
    usps     16.72x     3.00x   [19.53, 14.81, 12.5, 19.17, 17.57]

M_FIXED per seed: {0: 500, 1: 1000, 2: 500, 3: 1000, 4: 250}


**Il rapporto sul source (SVHN, ~74.6x ± 9.6x) è coerente con il ~64x del notebook a singolo
seed** (seed=2019 rientra nel range osservato qui, 66-88x) -- più stabile fra seed di quanto
osservato su Amazon Reviews (dove la std era quasi grande quanto la media). Il rapporto sui
target invece è molto più basso: mnist ~7.2x ± 1.8x, usps ~16.7x ± 3.0x -- entrambi ben sotto
il rapporto del source, un pattern diverso da Amazon Reviews (dove il rapporto cresceva
leggermente dal source ai target) -- qui l'epistemica del source cresce relativamente di più
del previsto quando valutata su MNIST/USPS, restringendo il rapporto rispetto al source
stesso, pur restando sempre aleatoria-dominante.

## 5. Tabelle (una per target): accuracy pre/post/delta, media ± std sui 2 bracci

In [ ]:
ARMS = ["shot_im", "u_sfan"]
TARGETS = ["mnist", "usps"]

summary = {}
for t in TARGETS:
    summary[t] = {}
    print(f"=== target: {t} ===")
    print(f"{'braccio':>10s} {'pre':>16s} {'post':>16s} {'delta':>18s}")
    print("-" * 64)
    for a in ARMS:
        pre = np.array([results_per_seed[s]["adaptation"][t][a]["acc_pre"] for s in SEEDS])
        post = np.array([results_per_seed[s]["adaptation"][t][a]["acc_post"] for s in SEEDS])
        delta = post - pre
        summary[t][a] = dict(pre=pre, post=post, delta=delta)
        print(f"{a:>10s} {100*pre.mean():6.2f}%+-{100*pre.std(ddof=1):4.2f} "
              f"{100*post.mean():6.2f}%+-{100*post.std(ddof=1):4.2f} "
              f"{100*delta.mean():+7.2f}pp+-{100*delta.std(ddof=1):5.2f}")
    print()

=== target: mnist ===
   braccio              pre             post              delta
----------------------------------------------------------------
   shot_im  60.14%+-2.91  76.03%+-4.19   +15.89pp+- 5.69
    u_sfan  60.14%+-2.91  73.28%+-7.12   +13.14pp+- 5.27

=== target: usps ===
   braccio              pre             post              delta
----------------------------------------------------------------
   shot_im  60.28%+-2.23  58.21%+-16.43   -2.07pp+-18.16
    u_sfan  60.28%+-2.23  46.44%+-22.11  -13.84pp+-24.26



**Su usps la deviazione standard supera in valore assoluto la media stessa** (18.16pp e
24.26pp di std, contro delta medi di -2.07pp e -13.84pp): un segnale inequivocabile che
l'adattamento su questo target, con questi iperparametri, non ha un effetto stabile --
oscilla fra miglioramenti sostanziali e collassi catastrofici a seconda del seed (si veda la
Sezione 3: seed 0 arriva a -54.21pp per u_sfan). Su mnist il quadro è più stabile ma comunque
rumoroso (std ~5-7pp su medie ~13-16pp).

## 6. Wilcoxon signed-rank, accoppiato per seed

In [ ]:
from scipy.stats import wilcoxon

for t in TARGETS:
    d_shot, d_usfan = summary[t]["shot_im"]["delta"], summary[t]["u_sfan"]["delta"]
    stat, p = wilcoxon(d_shot, d_usfan)
    print(f"=== {t} ===")
    print(f"  shot_im vs u_sfan:          W={stat:.1f}  p={p:.4f}")
    print(f"    differenze (pp), una per seed: {[round(100*x,2) for x in (d_shot-d_usfan)]}")
    print()

=== mnist ===
  shot_im vs u_sfan:          W=5.0  p=0.6250
    differenze (pp), una per seed: [-5.84, 2.82, -5.12, 10.32, 11.57]

=== usps ===
  shot_im vs u_sfan:          W=2.0  p=0.1875
    differenze (pp), una per seed: [23.37, 18.83, -16.09, 22.27, 10.46]



Nessun confronto raggiunge la significatività convenzionale. Su **mnist** il segno è
inconsistente (3/5 seed a favore di shot_im, 2/5 a favore di u_sfan) -- non c'è un vincitore
chiaro. Su **usps** il segno è più consistente (4/5 seed a favore di shot_im, `W=2.0`, `p=
0.1875` -- non lontanissimo dalla soglia convenzionale) ma la dimensione dell'effetto varia
enormemente (da +23.37pp a -16.09pp): anche quando la direzione è la stessa, la magnitudo
non lo è, coerente con l'enorme varianza già osservata in Sezione 5.

## 7. Confronto esplicito: singolo seed (11_digits_shift_adapt.ipynb) vs. media 5 seed

| target | delta shot_im (1 seed) | delta u_sfan (1 seed) | delta shot_im (media 5 seed) | delta u_sfan (media 5 seed) | pattern confermato? | p-value Wilcoxon |
|---|---|---|---|---|---|---|
| mnist | +19.78pp | +15.80pp | +15.89pp ± 5.69 | +13.14pp ± 5.27 | **parzialmente** -- direzione media sì, ma non significativo e invertito in 2/5 seed | 0.625 |
| usps | +9.72pp | +3.64pp | **-2.07pp ± 18.16** | **-13.84pp ± 24.26** | **no** -- il segno medio si INVERTE (entrambi i bracci peggiorano l'accuracy in media), varianza enorme | 0.1875 |

**Il pattern "shot_im batte u_sfan su entrambi i target" non si dissolve nel rumore come su
Amazon Reviews -- fa qualcosa di più drastico su usps: il segno stesso del beneficio
dell'adattamento si inverte.** Sul singolo seed (2019), l'adattamento aiutava sempre, di più
con shot_im che con u_sfan -- un quadro pulito e incoraggiante. Sui 5 seed, quel seed si
rivela essere stato tra i più favorevoli: la media reale è che l'adattamento su usps, con
questi iperparametri (`lr=1e-2`, `steps=50`, full batch), è tanto probabile che aiuti quanto
che *danneggi* gravemente il modello (fino a -54pp in un caso), un rischio invisibile
nell'unico seed originariamente riportato. Su mnist il quadro è meno drammatico ma comunque
più debole e meno chiaro di quanto il singolo seed suggerisse: shot_im resta in media
davanti, ma non in modo statisticamente distinguibile, e l'ordine si inverte in 2 run su 5.

## 8. Collocazione nel confronto multi-esperimento del progetto, e aggiornamento della discussione di letteratura

| esperimento | rapporto aleatoria/epistemica (source) | vince (1 seed) | vince (media 5 seed) | validazione multi-seed |
|---|---|---|---|---|
| SVHN -> MNIST/USPS (questo esperimento) | ~74.6x ± 9.6x (source) | shot_im su entrambi | **nessuno in modo affidabile** -- mnist non significativo, usps segno invertito in media | completa (source + adattamento) |
| Electronics -> dvd/kitchen/books (Amazon Reviews) | ~95.9x-99.7x | u_sfan su tutti e 3 | u_sfan su tutti e 3 (confermato) | completa (source + adattamento) |

**Aggiornamento della discussione di Kendall & Gal (2017).** La discussione in
`11_digits_shift_adapt.ipynb` interpretava il vantaggio di shot_im su SVHN (rapporto
aleatoria/epistemica ~64x) come una conferma diretta del meccanismo di Kendall & Gal: un
source con poca epistemica residua rende il peso a entropia totale di U-SFAN controproducente,
perché pesa per l'aleatoria mascherata da incertezza. **Il risultato multi-seed non smentisce
il meccanismo in sé** (il rapporto sul source, ~74.6x ± 9.6x, resta shot_im-favorevole in
media su entrambi i target) **ma mostra che l'osservazione puntuale a singolo seed non era
generalizzabile come "shot_im batte sempre u_sfan qui" -- va riformulata come "shot_im tende a
battere u_sfan in media, ma il risultato di un singolo run, specialmente su usps, non è
rappresentativo della distribuzione reale dei risultati possibili".** La differenza rispetto
ad Amazon Reviews è istruttiva: là il pattern (u_sfan vince) si è confermato con un margine
STABILE su tutti e 3 i target multi-seed; qui il pattern (shot_im vince) si INDEBOLISCE
drasticamente passando da un singolo seed a 5, al punto da invertirsi in media su un target.
Kendall & Gal predice correttamente la DIREZIONE attesa del meccanismo (source
aleatoria-dominante -> shot_im avvantaggiato), ma qui non basta a spiegare la MAGNITUDO reale
né, soprattutto, la sua enorme variabilità fra run -- un limite esplicito del confronto a
singolo seed che il notebook originale non poteva rivelare da solo.

**Nota conclusiva.** Questo è ora il secondo esperimento del progetto (dopo Amazon Reviews)
con validazione multi-seed completa. Nei due casi il verdetto "il pattern a singolo seed
regge?" è diverso -- confermato su Amazon Reviews, in gran parte no qui -- il che è di per sé
la lezione più importante: **non esiste un'aspettativa di default** ("il singolo seed
probabilmente regge" o "probabilmente no") applicabile a questo tipo di esperimento
(MLP/CNN piccola, IM adaptation full-batch); ogni pattern a singolo seed riportato nel resto
del progetto (MNIST-pieno -> Rotated-MNIST, in particolare) resta da verificare caso per
caso, non per analogia con uno di questi due risultati.